# OOP Week 12 -- Plugin Exercise

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-11
**Focus:** add a new analyzer WITHOUT touching core code

---

## Learning Objectives

1. Add a new component to an existing system without modification
2. Demonstrate OCP in practice
3. Write tests for the new component
4. Register the component with the factory
5. Verify the pipeline still works

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## The Challenge

You will add a **PercentileAnalyzer** to the pipeline. Rules:

1. Do NOT modify any existing class (AnalyzerBase, MeanAnalyzer, etc.)
2. Do NOT modify the AnalyzerFactory class
3. Do NOT modify any existing test
4. Write the new analyzer in a new class
5. Register it with the factory (ONE line)
6. Write at least 3 tests for it

This proves that the architecture is truly open for extension.

---
## Existing Code (DO NOT MODIFY)

In [ ]:
# ===== EXISTING CODE -- DO NOT MODIFY =====

class AnalyzerBase:
    def analyze(self, values):
        raise NotImplementedError

class MeanAnalyzer(AnalyzerBase):
    def analyze(self, values):
        return {"mean": round(sum(values)/len(values), 4)} if values else {}

class StdAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        m = sum(values)/len(values)
        return {"std": round((sum((x-m)**2 for x in values)/len(values))**0.5, 4)}

class EventAnalyzer(AnalyzerBase):
    def __init__(self, threshold=50):
        self.threshold = threshold
    def analyze(self, values):
        return {"events_above": sum(1 for v in values if v > self.threshold)}

class AnalyzerFactory:
    _registry = {}
    @classmethod
    def register(cls, name, klass):
        cls._registry[name] = klass
    @classmethod
    def create(cls, name, **kw):
        if name not in cls._registry:
            raise ValueError("Unknown: " + name)
        return cls._registry[name](**kw)
    @classmethod
    def list_available(cls):
        return list(cls._registry.keys())

# Register existing
AnalyzerFactory.register("mean", MeanAnalyzer)
AnalyzerFactory.register("std", StdAnalyzer)
AnalyzerFactory.register("events", EventAnalyzer)

print("Existing analyzers:", AnalyzerFactory.list_available())
# ===== END EXISTING CODE =====

**Expected Output:**
```
Existing analyzers: ['mean', 'std', 'events']
```

---
## Your Task: Add PercentileAnalyzer

Create a `PercentileAnalyzer` class that:
- Inherits from `AnalyzerBase`
- Takes a list of percentiles (default: [25, 50, 75])
- Returns dict with keys like `p25`, `p50`, `p75`
- Handles empty input gracefully

In [ ]:
# Step 1: Create the class (inherits from AnalyzerBase)
# YOUR CODE HERE

# Step 2: Register with factory (ONE line)
# YOUR CODE HERE

# Step 3: Test it
# YOUR CODE HERE

---
## Solution (try yourself first!)

In [ ]:
class PercentileAnalyzer(AnalyzerBase):
    """Computes percentiles of numeric values."""
    def __init__(self, percentiles=None):
        self.percentiles = percentiles or [25, 50, 75]

    def analyze(self, values):
        if not values:
            return {}
        s = sorted(values)
        n = len(s)
        result = {}
        for p in self.percentiles:
            idx = min(int(n * p / 100), n - 1)
            result["p" + str(p)] = s[idx]
        return result

# Register (ONE line!)
AnalyzerFactory.register("percentile", PercentileAnalyzer)

print("Available:", AnalyzerFactory.list_available())

# Create from factory
pa = AnalyzerFactory.create("percentile", percentiles=[10, 50, 90])
values = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
print("Results:", pa.analyze(values))

**Expected Output:**
```
Available: ['mean', 'std', 'events', 'percentile']
Results: {'p10': 20, 'p50': 60, 'p90': 100}
```

---
## Tests for PercentileAnalyzer

In [ ]:
def test_percentile_normal():
    pa = PercentileAnalyzer()
    result = pa.analyze([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
    assert "p25" in result
    assert "p50" in result
    assert "p75" in result

def test_percentile_empty():
    pa = PercentileAnalyzer()
    assert pa.analyze([]) == {}

def test_percentile_custom():
    pa = PercentileAnalyzer(percentiles=[50])
    result = pa.analyze([1, 2, 3, 4, 5])
    assert "p50" in result
    assert "p25" not in result  # only asked for 50

def test_percentile_factory():
    pa = AnalyzerFactory.create("percentile")
    assert isinstance(pa, PercentileAnalyzer)

test_percentile_normal()
print("[PASS] test_percentile_normal")
test_percentile_empty()
print("[PASS] test_percentile_empty")
test_percentile_custom()
print("[PASS] test_percentile_custom")
test_percentile_factory()
print("[PASS] test_percentile_factory")
print()
print("All plugin tests passed!")
print("And we did NOT modify any existing code!")

**Expected Output:**
```
[PASS] test_percentile_normal
[PASS] test_percentile_empty
[PASS] test_percentile_custom
[PASS] test_percentile_factory

All plugin tests passed!
And we did NOT modify any existing code!
```

---
## Mini-Quiz

In [ ]:
# Q1: Which principle did we demonstrate? (SRP, OCP, or both?)
# Answer: 

# Q2: How many existing files did we modify?
# Answer: 

# Q3: What made this possible? (what patterns?)
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)